# Vicon vs Awinda IMU kinematics — unassisted walking

Paper section *Influence of Kinematic Input Quality* (Awinda / full lower-limb).

Replicates the evaluation shell of `process_awinda.ipynb` / `visualize_awinda.ipynb` on the same trials, with **only the kinematic inputs swapped**:

| Arm | Inputs | Estimator |
|-----|--------|-----------|
| **Awinda (IMU IK)** | `mt_processed/.../ik/VQF/{speed}_{cond}.pkl` | cached `pred_nmpkg` from `process_awinda.npz` |
| **Vicon IK** | `processed/.../awinda/ik/{COND}_{speed}_ik.mot` | same `0512_ik_id_all_zero_in_zero_out` checkpoint offline |

**Shared with process_awinda (unchanged):**
- Checkpoint, filters (zero-phase 6 Hz angle/output, 15 Hz velocity), bilateral TCN
- Sync lag from Awinda↔Vicon angle xcorr (stored in cache meta)
- GT = OpenSim ID / mass, same post-sync **15 s** transient trim
- Paper exclusion: `AB05_Maria::LG_0p8mps` (matches `visualize_awinda.ipynb`)

Set `FORCE_REPLAY = True` to re-run Vicon→TCN; `False` loads the metrics CSV.



In [ ]:
import io
import json
import pickle
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, sosfilt, sosfiltfilt

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN

PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
CHECKPOINT = PROJECT_ROOT / 'runs' / '0512_ik_id_all_zero_in_zero_out' / 'best_model.pt'
AWINDA_CACHE = PROJECT_ROOT / 'analysis' / 'cache' / 'process_awinda.npz'
OUT_DIR = PROJECT_ROOT / 'analysis' / 'paper_outputs' / 'kinematic_input_quality'
FIG_DIR = OUT_DIR / 'figures'
METRICS_CSV = PROJECT_ROOT / 'analysis' / 'cache' / 'vicon_vs_awinda_metrics.csv'
DETAIL_CSV = PROJECT_ROOT / 'analysis' / 'cache' / 'vicon_vs_awinda_metrics_by_channel.csv'
KIN_CSV = PROJECT_ROOT / 'analysis' / 'cache' / 'vicon_vs_awinda_kinematic_corr.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FORCE_REPLAY = False

SUBJECT_MASS_KG = {
    'AB01_Jinwoo': 88.0, 'AB02_Oscar': 71.1, 'AB03_Ilseung': 84.4,
    'AB04_Changseob': 74.0, 'AB05_Maria': 55.0, 'AB06_Jimin': 82.6,
    'AB07_Amy': 51.3, 'AB08_Seokhyun': 71.9,
}
CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]
ID_COLS = [f'{c}_moment' for c in CHANNELS]
IK_CHANNEL_IDX = [IK_DOF_NAMES.index(c) for c in CHANNELS]
DEFAULT_FS_HZ = 100.0
TRANSIENT_TRIM_SEC = 15.0
PAPER_EXCLUDE_TRIALS = {'AB05_Maria::LG_0p8mps'}

JOINT_LABELS = {
    'hip_flexion_r': 'Hip R', 'knee_angle_r': 'Knee R', 'ankle_angle_r': 'Ankle R',
    'hip_flexion_l': 'Hip L', 'knee_angle_l': 'Knee L', 'ankle_angle_l': 'Ankle L',
}

REPLAY_WAVES: Dict[str, Dict] = {}


def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, x) if mode == 'zero_phase' else sosfilt(sos, x)


def lpf_mc(X, fs_hz, cutoff_hz, order, mode='zero_phase'):
    return np.column_stack([
        butter_lpf(X[:, c], fs_hz, cutoff_hz, order, mode) for c in range(X.shape[1])
    ])


def fill_nan_cols(X: np.ndarray) -> np.ndarray:
    out = np.asarray(X, dtype=np.float64).copy()
    for c in range(out.shape[1]):
        col = out[:, c]
        finite = np.isfinite(col)
        if finite.all() or not finite.any():
            continue
        out[~finite, c] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), col[finite])
    return out


def condition_to_pkl_stem(condition: str) -> str:
    cond, speed = condition.split('_', 1)
    return f'{speed}_{cond.lower()}'


def resolve_trial_paths(subject: str, condition: str) -> Dict[str, Path]:
    pkl_stem = condition_to_pkl_stem(condition)
    ik_dir = IMU_IK_ROOT / subject / 'ik' / IMU_IK_METHOD
    if not ik_dir.exists():
        ik_dir = IMU_IK_ROOT / subject
    return {
        'pkl': ik_dir / f'{pkl_stem}.pkl',
        'id': PROCESSED_ROOT / subject / 'awinda' / 'id' / f'{condition}_id.sto',
        'vicon': PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{condition}_ik.mot',
    }


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return pos_deg


def build_vicon_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def rmse_r2(y_pred, y_true):
    """RMSE and R² = (Pearson r)² — matches paper_outputs/awinda ID tables."""
    m = np.isfinite(y_pred) & np.isfinite(y_true)
    if m.sum() < 2:
        return np.nan, np.nan
    yp = y_pred[m]
    yt = y_true[m]
    rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
    r = float(np.corrcoef(yp, yt)[0, 1])
    return rmse, float(r * r)


def corr_1d(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 100:
        return np.nan
    return float(np.corrcoef(a[m], b[m])[0, 1])


def _trial_key_to_prefix(trial_key: str) -> str:
    return trial_key.replace('::', '__')


def load_awinda_cache():
    data = np.load(str(AWINDA_CACHE), allow_pickle=True)
    trial_keys = [str(k) for k in data['trial_keys']]
    waves = {}
    for trial_key in trial_keys:
        p = _trial_key_to_prefix(trial_key)
        meta = data[f'{p}__meta']
        waves[trial_key] = {
            'subject': str(meta[0]),
            'condition': str(meta[1]),
            'mass_kg': float(meta[2]),
            'lag_samples': int(meta[3]),
            'lag_seconds': float(meta[4]),
            'xcorr_score': float(meta[5]),
            'lag_clipped': bool(meta[6]),
            't': np.asarray(data[f'{p}__t'], dtype=np.float64),
            'pred_awinda': np.asarray(data[f'{p}__pred_nmpkg'], dtype=np.float64),
            'id_nmpkg': np.asarray(data[f'{p}__id_nmpkg'], dtype=np.float64),
            'rmse_awinda': np.asarray(data[f'{p}__rmse_nmpkg'], dtype=np.float64),
            'r2_awinda': np.asarray(data[f'{p}__r2_nmpkg'], dtype=np.float64),
        }
    return waves, data


def load_model():
    ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
    with open(CHECKPOINT.parent / 'config.json') as f:
        train_cfg = json.load(f)
    cfg = ckpt['model_config']
    model = TCN(**{k: cfg[k] for k in [
        'n_input_channels', 'n_output_channels', 'hidden_channels', 'n_blocks', 'kernel_size', 'dropout'
    ]})
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    window = int(ckpt.get('window_size', 100))
    input_indices = list(ckpt.get('input_indices', [6, 9, 10, 13, 16, 17]))
    h = len(input_indices) // 2
    filters = {
        'angle_cutoff': float(train_cfg.get('lowpass_cutoff_hz', 6.0)),
        'vel_cutoff': float(train_cfg.get('velocity_lowpass_cutoff_hz') or 15.0),
        'out_cutoff': float(train_cfg.get('lowpass_cutoff_hz', 6.0)),
        'order': int(train_cfg.get('lowpass_order', 4)),
        'in_mode': str(train_cfg.get('input_lowpass_mode', 'zero_phase')),
        'out_mode': str(train_cfg.get('output_lowpass_mode', 'zero_phase')),
    }
    return model, cfg, window, input_indices[:h], input_indices[h:], filters


@torch.no_grad()
def infer_one_side(model, cfg, window, pos_3, vel_3):
    x = np.concatenate([pos_3, vel_3], axis=1).astype(np.float32)
    n, c_out = x.shape[0], cfg['n_output_channels']
    pred = np.zeros((n, c_out), dtype=np.float64)

    def _fwd(start):
        xt = torch.from_numpy(np.ascontiguousarray(x[start:start + window].T)).unsqueeze(0).to(DEVICE)
        return model(xt).squeeze(0).detach().cpu().numpy().T

    pred[:window] = _fwd(0)
    for start in range(1, n - window + 1):
        pred[start + window - 1] = _fwd(start)[window - 1]
    return pred.astype(np.float32)


def run_bilateral(model, cfg, window, idx_r, idx_l, pos_full, vel_full):
    pr = infer_one_side(model, cfg, window, pos_full[:, idx_r], vel_full[:, idx_r])
    pl = infer_one_side(model, cfg, window, pos_full[:, idx_l], vel_full[:, idx_l])
    return np.concatenate([pr, pl], axis=1)


def replay_vicon_one(trial_key: str, wave: Dict, model, cfg, window, idx_r, idx_l, filters,
                     *, store_wave: bool = True) -> Tuple[Dict, List[Dict], Dict]:
    subject, condition = wave['subject'], wave['condition']
    paths = resolve_trial_paths(subject, condition)
    mass = wave['mass_kg']
    lag = int(wave['lag_samples'])

    id_df = parse_opensim_table(paths['id'])
    t_id = id_df.index.to_numpy(dtype=np.float64)
    fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ

    id_nm = np.column_stack([
        id_df[c].to_numpy(dtype=np.float64) if c in id_df.columns else np.full(len(t_id), np.nan)
        for c in ID_COLS
    ])
    id_full = lpf_mc(id_nm / mass, fs_hz, filters['out_cutoff'], filters['order'], filters['out_mode'])

    vicon_rad = fill_nan_cols(build_vicon_ik_rad(parse_opensim_table(paths['vicon'])))
    pos_f = lpf_mc(vicon_rad, fs_hz, filters['angle_cutoff'], filters['order'], filters['in_mode'])
    vel_f = lpf_mc(np.gradient(pos_f, 1.0 / fs_hz, axis=0), fs_hz, filters['vel_cutoff'], filters['order'], filters['in_mode'])
    pred_full = lpf_mc(
        run_bilateral(model, cfg, window, idx_r, idx_l, pos_f, vel_f),
        fs_hz, filters['out_cutoff'], filters['order'], filters['out_mode'],
    )

    # Same sync window as process_awinda: ID/Vicon share mocap clock; Awinda shifted by lag.
    start_id = max(-lag, 0)
    trim_n = int(round(TRANSIENT_TRIM_SEC * fs_hz))
    n_cached = len(wave['id_nmpkg'])
    n_sync = n_cached + trim_n
    if start_id + n_sync > min(len(id_full), len(pred_full)):
        n_sync = min(len(id_full), len(pred_full)) - start_id
        n_cached = max(0, n_sync - trim_n)

    sl = slice(start_id + trim_n, start_id + trim_n + n_cached)
    pred_vicon = np.asarray(pred_full[sl], dtype=np.float64)
    id_ref = np.asarray(wave['id_nmpkg'][:n_cached], dtype=np.float64)
    pred_awinda = np.asarray(wave['pred_awinda'][:n_cached], dtype=np.float64)
    n = min(len(pred_awinda), len(pred_vicon), len(id_ref))
    pred_awinda, pred_vicon, id_ref = pred_awinda[:n], pred_vicon[:n], id_ref[:n]

    # Kinematic corr on synced angles (Awinda PKL vs Vicon), using same lag alignment
    imu = pickle.load(open(paths['pkl'], 'rb'))
    awinda_rad = np.deg2rad(build_model_input_from_pkl(imu))
    # Apply same offsets as process_awinda for fair angle corr
    # (offsets live only on IMU path; imported inline to avoid drift)
    ANGLE_OFFSET_DEG = {
        ('AB02_Oscar', 'LG_0p8mps'): {c: 6.0 for c in CHANNELS},
        ('AB01_Jinwoo', 'RD_0p8mps'): {
            'knee_angle_r': 10.0, 'knee_angle_l': 10.0,
            'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0,
        },
    }
    offsets = ANGLE_OFFSET_DEG.get((subject, condition))
    if offsets:
        for name, deg in offsets.items():
            awinda_rad[:, IK_DOF_NAMES.index(name)] += np.deg2rad(float(deg))
    awinda_f = lpf_mc(awinda_rad[:, IK_CHANNEL_IDX], fs_hz, filters['angle_cutoff'], filters['order'], filters['in_mode'])
    vicon_f = lpf_mc(vicon_rad[:, IK_CHANNEL_IDX], fs_hz, filters['angle_cutoff'], filters['order'], filters['in_mode'])
    start_a, start_v = max(lag, 0), max(-lag, 0)
    nn = min(len(awinda_f) - start_a, len(vicon_f) - start_v) - trim_n
    aw_ang = awinda_f[start_a + trim_n:start_a + trim_n + nn]
    vi_ang = vicon_f[start_v + trim_n:start_v + trim_n + nn]

    detail_rows = []
    kin_row = {'trial': trial_key, 'subject': subject, 'condition': condition}
    for c, ch in enumerate(CHANNELS):
        rmse_a, r2_a = rmse_r2(pred_awinda[:, c], id_ref[:, c])
        rmse_v, r2_v = rmse_r2(pred_vicon[:, c], id_ref[:, c])
        detail_rows.append({
            'trial': trial_key, 'subject': subject, 'condition': condition, 'channel': ch,
            'rmse_awinda_nmpkg': rmse_a, 'r2_awinda': r2_a,
            'rmse_vicon_nmpkg': rmse_v, 'r2_vicon': r2_v,
            'rmse_reduction_pct': float(100.0 * (rmse_a - rmse_v) / rmse_a) if rmse_a and rmse_a > 0 else np.nan,
            'r2_increase': float(r2_v - r2_a) if np.isfinite(r2_v) and np.isfinite(r2_a) else np.nan,
            'corr_angle': corr_1d(aw_ang[:, c], vi_ang[:, c]) if c < aw_ang.shape[1] else np.nan,
        })
        kin_row[f'corr_{ch}'] = detail_rows[-1]['corr_angle']

    mean_rmse_a = float(np.nanmean([r['rmse_awinda_nmpkg'] for r in detail_rows]))
    mean_r2_a = float(np.nanmean([r['r2_awinda'] for r in detail_rows]))
    mean_rmse_v = float(np.nanmean([r['rmse_vicon_nmpkg'] for r in detail_rows]))
    mean_r2_v = float(np.nanmean([r['r2_vicon'] for r in detail_rows]))
    mean_corr = float(np.nanmean([r['corr_angle'] for r in detail_rows]))

    summary = {
        'trial': trial_key,
        'subject': subject,
        'condition': condition,
        'task': condition.split('_')[0],
        'lag_samples': lag,
        'n': n,
        'rmse_awinda_nmpkg': mean_rmse_a,
        'r2_awinda': mean_r2_a,
        'rmse_vicon_nmpkg': mean_rmse_v,
        'r2_vicon': mean_r2_v,
        'rmse_reduction_pct': float(100.0 * (mean_rmse_a - mean_rmse_v) / mean_rmse_a) if mean_rmse_a > 0 else np.nan,
        'r2_increase': float(mean_r2_v - mean_r2_a),
        'mean_angle_corr': mean_corr,
    }
    kin_row['mean_angle_corr'] = mean_corr

    if store_wave:
        hip_i = CHANNELS.index('hip_flexion_r')
        REPLAY_WAVES[trial_key] = {
            't': wave['t'][:n].copy(),
            'id': id_ref[:, hip_i],
            'awinda': pred_awinda[:, hip_i],
            'vicon': pred_vicon[:, hip_i],
            'summary': summary,
        }
    return summary, detail_rows, kin_row


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    groups = [('overall', df)]
    for task in sorted(df['task'].unique()):
        groups.append((task, df[df['task'] == task]))
    for label, sub in groups:
        if sub.empty:
            continue
        rows.append({
            'group': label,
            'n_trials': int(len(sub)),
            'rmse_awinda_nmpkg': float(sub['rmse_awinda_nmpkg'].mean()),
            'r2_awinda': float(sub['r2_awinda'].mean()),
            'rmse_vicon_nmpkg': float(sub['rmse_vicon_nmpkg'].mean()),
            'r2_vicon': float(sub['r2_vicon'].mean()),
            'rmse_reduction_pct': float(sub['rmse_reduction_pct'].mean()),
            'r2_increase': float(sub['r2_increase'].mean()),
            'mean_angle_corr': float(sub['mean_angle_corr'].mean()),
        })
    return pd.DataFrame(rows)


def write_paper(summary: pd.DataFrame, per_trial: pd.DataFrame, detail: pd.DataFrame):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    overall = summary.set_index('group').loc['overall']

    joint_rows = []
    for ch in CHANNELS:
        sub = detail[detail['channel'] == ch]
        ra, rv = sub['rmse_awinda_nmpkg'], sub['rmse_vicon_nmpkg']
        ca, cv = sub['r2_awinda'], sub['r2_vicon']
        joint_rows.append({
            'joint': JOINT_LABELS[ch],
            'n_trials': int(len(sub)),
            'rmse_awinda': f'{ra.mean():.3f} ± {ra.std(ddof=1):.3f}',
            'r2_awinda': f'{ca.mean():.3f} ± {ca.std(ddof=1):.3f}',
            'rmse_vicon': f'{rv.mean():.3f} ± {rv.std(ddof=1):.3f}',
            'r2_vicon': f'{cv.mean():.3f} ± {cv.std(ddof=1):.3f}',
            'rmse_reduction_pct': float(100.0 * (ra.mean() - rv.mean()) / ra.mean()),
            'r2_increase': float(cv.mean() - ca.mean()),
            'angle_corr': float(sub['corr_angle'].mean()),
            '_rmse_a': float(ra.mean()), '_rmse_a_std': float(ra.std(ddof=1)),
            '_r2_a': float(ca.mean()), '_r2_a_std': float(ca.std(ddof=1)),
            '_rmse_v': float(rv.mean()), '_rmse_v_std': float(rv.std(ddof=1)),
            '_r2_v': float(cv.mean()), '_r2_v_std': float(cv.std(ddof=1)),
        })
    joint_df = pd.DataFrame(joint_rows)
    joint_csv = OUT_DIR / 'awinda_per_joint.csv'
    joint_df.drop(columns=[c for c in joint_df.columns if c.startswith('_')]).to_csv(joint_csv, index=False)
    print(f'Wrote {joint_csv}')
    print(joint_df[['joint', 'rmse_awinda', 'r2_awinda', 'rmse_vicon', 'r2_vicon', 'rmse_reduction_pct', 'r2_increase']].to_string(index=False))

    snip = OUT_DIR / 'awinda_paper_snippet.tex'
    lines = [
        '% Awinda — kinematic input quality (IMU IK vs Vicon IK)',
        '% R² = (Pearson r)^2 (matches paper_outputs/awinda table1/table3/table6).',
        f'Awinda IMU IK: RMSE {overall["rmse_awinda_nmpkg"]:.3f} ± {per_trial["rmse_awinda_nmpkg"].std(ddof=1):.3f}~Nm/kg, '
        f'$R^2$ {overall["r2_awinda"]:.3f} ± {per_trial["r2_awinda"].std(ddof=1):.3f}.',
        f'Vicon IK substitution (same full-limb TCN): RMSE {overall["rmse_vicon_nmpkg"]:.3f} ± {per_trial["rmse_vicon_nmpkg"].std(ddof=1):.3f}~Nm/kg, '
        f'$R^2$ {overall["r2_vicon"]:.3f} ± {per_trial["r2_vicon"].std(ddof=1):.3f}',
        f'({overall["rmse_reduction_pct"]:.1f}\% RMSE reduction vs Awinda, absolute $R^2$ increase {overall["r2_increase"]:+.3f}).',
        f'Mean Awinda–Vicon angle correlation (6 DOF): {overall["mean_angle_corr"]:.3f}.',
        '',
        '% Per-joint (mean ± std over trials):',
    ]
    for r in joint_rows:
        lines.append(
            f'% {r["joint"]}: Awinda RMSE {r["_rmse_a"]:.3f}±{r["_rmse_a_std"]:.3f}, '
            f'$R^2$ {r["_r2_a"]:.3f}±{r["_r2_a_std"]:.3f}; '
            f'Vicon RMSE {r["_rmse_v"]:.3f}±{r["_rmse_v_std"]:.3f}, '
            f'$R^2$ {r["_r2_v"]:.3f}±{r["_r2_v_std"]:.3f} '
            f'(RMSE {r["rmse_reduction_pct"]:+.1f}\%, $R^2$ {r["r2_increase"]:+.3f}).'
        )
    snip.write_text('\n'.join(lines) + '\n')
    print(f'Wrote {snip}')
    print(snip.read_text())


# ---- main ----
if not AWINDA_CACHE.is_file():
    raise FileNotFoundError(f'Missing {AWINDA_CACHE}. Run process_awinda.ipynb first.')

waves, _cache = load_awinda_cache()
trial_keys = [k for k in waves if k not in PAPER_EXCLUDE_TRIALS]
print(f'Awinda cache: {len(waves)} trials | paper subset: {len(trial_keys)} (exclude {sorted(PAPER_EXCLUDE_TRIALS)})')

if (not FORCE_REPLAY) and METRICS_CSV.is_file() and DETAIL_CSV.is_file():
    print(f'Loading cached metrics from {METRICS_CSV}')
    metrics_df = pd.read_csv(METRICS_CSV)
    detail_df = pd.read_csv(DETAIL_CSV)
    kin_df = pd.read_csv(KIN_CSV) if KIN_CSV.is_file() else pd.DataFrame()
else:
    model, cfg, window, idx_r, idx_l, filters = load_model()
    print(f'Checkpoint: {CHECKPOINT}')
    print(
        f'Filters: angle={filters["angle_cutoff"]}Hz/{filters["in_mode"]}, '
        f'vel={filters["vel_cutoff"]}Hz, out={filters["out_cutoff"]}Hz/{filters["out_mode"]} | trim={TRANSIENT_TRIM_SEC:g}s'
    )
    REPLAY_WAVES.clear()
    summaries, details, kins = [], [], []
    for trial_key in sorted(trial_keys):
        summary, detail_rows, kin_row = replay_vicon_one(
            trial_key, waves[trial_key], model, cfg, window, idx_r, idx_l, filters,
        )
        summaries.append(summary)
        details.extend(detail_rows)
        kins.append(kin_row)
        print(
            f"  {trial_key}: Awinda RMSE={summary['rmse_awinda_nmpkg']:.3f} R²={summary['r2_awinda']:.3f} | "
            f"Vicon RMSE={summary['rmse_vicon_nmpkg']:.3f} R²={summary['r2_vicon']:.3f} | "
            f"ang corr={summary['mean_angle_corr']:.3f}"
        )
    metrics_df = pd.DataFrame(summaries)
    detail_df = pd.DataFrame(details)
    kin_df = pd.DataFrame(kins)
    METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(METRICS_CSV, index=False)
    detail_df.to_csv(DETAIL_CSV, index=False)
    kin_df.to_csv(KIN_CSV, index=False)
    print(f'Wrote {METRICS_CSV} ({len(metrics_df)} trials)')

summary = summarize(metrics_df)
print('\n=== Awinda IMU IK vs Vicon IK substitution (R² = corr², paper formula) ===')
print(summary.to_string(index=False))
print('\n=== Per-channel means ===')
print(
    detail_df.groupby('channel')[['rmse_awinda_nmpkg', 'r2_awinda', 'rmse_vicon_nmpkg', 'r2_vicon', 'corr_angle']]
    .mean().reindex(CHANNELS).to_string()
)
write_paper(summary, metrics_df, detail_df)
metrics_df.head()








In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
PALETTE = {'gt': '#1e88e5', 'awinda': '#E65100', 'vicon': '#2e7d32'}


def _ensure_waves(stems: List[str]) -> None:
    missing = [s for s in stems if s not in REPLAY_WAVES]
    if not missing:
        return
    model, cfg, window, idx_r, idx_l, filters = load_model()
    waves_local, _ = load_awinda_cache()
    for stem in missing:
        replay_vicon_one(stem, waves_local[stem], model, cfg, window, idx_r, idx_l, filters, store_wave=True)


def plot_hip_timeseries(trial_key: str, t_window=(20.0, 35.0)) -> None:
    _ensure_waves([trial_key])
    w = REPLAY_WAVES[trial_key]
    t_rel = w['t'] - np.nanmin(w['t'])
    m = (t_rel >= t_window[0]) & (t_rel <= t_window[1])
    s = w['summary']
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.plot(t_rel[m], w['id'][m], color=PALETTE['gt'], lw=1.8, label='OpenSim ID')
    ax.plot(t_rel[m], w['awinda'][m], color=PALETTE['awinda'], lw=1.4, ls='--',
            label=f"Awinda  RMSE={s['rmse_awinda_nmpkg']:.3f} R²={s['r2_awinda']:.3f}")
    ax.plot(t_rel[m], w['vicon'][m], color=PALETTE['vicon'], lw=1.5,
            label=f"Vicon IK→TCN  RMSE={s['rmse_vicon_nmpkg']:.3f} R²={s['r2_vicon']:.3f}")
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Hip R moment (N·m/kg)')
    ax.set_title(trial_key)
    ax.legend(fontsize=8, loc='upper right')
    ax.axhline(0, color='0.7', lw=0.6)
    fig.tight_layout()
    out = FIG_DIR / f'{trial_key.replace("::", "__")}_awinda_vs_vicon_hip.png'
    fig.savefig(out, dpi=150)
    plt.show()
    print(f'Wrote {out}')


ranked = metrics_df.sort_values('r2_vicon')
exemplars = [ranked.iloc[-1]['trial'], ranked.iloc[len(ranked) // 2]['trial'], ranked.iloc[0]['trial']]
print('Exemplars (best / mid / worst Vicon mean R²):', exemplars)
for trial_key in exemplars:
    plot_hip_timeseries(trial_key)



## Notes

- **Awinda arm** is exactly `process_awinda.ipynb`: IMU IK → same full-limb TCN → OpenSim ID / mass.
- **Vicon arm** keeps that GT, sync lag, trim, and filters; only the kinematic input is replaced with Vicon IK.
- Angle offsets (`ANGLE_OFFSET_DEG`) apply to the Awinda IMU path only (as in `process_awinda`).
- Both arms use the **same offline `.pt` checkpoint**, so this comparison is fairer than hip-exo logged-TRT vs offline Vicon.
- Mean Awinda–Vicon **angle correlation** is reported per trial/channel as a sanity check that kinematics agree after xcorr sync.

- **R² formula**: squared Pearson correlation between prediction and OpenSim ID (same as `paper_outputs/awinda` table1 / table3 / table6). This differs from CoD \(R^2=1-\mathrm{SSE}/\mathrm{SST}\) used in `process_awinda` cache `r2_nmpkg`.

